In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import scipy.stats as stats
import matplotlib.pyplot as plt

In [2]:
update_prof = {'Episódico/Epidêmico':'Episodic/Epidemic', 'Epidêmico':'Epidemic', 'Episódico':'Episodic',
       'Transmissão Persistente':'Persistent Transmission', 'Sem transmissão': 'No Transmission'}
prof_order = ['No Transmission', 'Episodic', 'Episodic/Epidemic', 'Epidemic', 
       'Persistent Transmission']


#### Tabela com as variáveis categóricas

Abrindo o dataframe com os perfis: 

In [3]:
df_perfis = pd.read_csv('../Dengue/dengue_pattern_10_19.csv', sep = ';', usecols = ['muni_code', 'dengue_pattern'])

df_perfis.head()

,muni_code,dengue_pattern
0,1100015,Episódico/Epidêmico
1,1100023,Epidêmico
2,1100031,Episódico/Epidêmico
3,1100049,Epidêmico
4,1100056,Episódico/Epidêmico


Abrindo o dataframe com os perfis e fazendo o merge deles: 

In [4]:
df_det = pd.read_csv('../Determinantes/banco_determinantes_fim.csv',encoding='latin-1', sep = ';')

df_det = df_det.merge(df_perfis, left_on='muni_code', right_on = 'muni_code')

df_det.loc[:, 'dengue_pattern'] = df_det.dengue_pattern.replace(update_prof)

for col in ['altitude', 'prop_esgot', 'prop_abast', 'prop_residuo']:
    df_det[col] = df_det[col].str.replace(',', '.').astype(float)

df_det.head()

,muni_code,muni_name,uf_code,uf_acr,tipo_regiao,bioma,altitude,koppen,pop22,prop_esgot,prop_abast,prop_residuo,dengue_pattern
0,1100015,Alta Floresta D'oeste,11,RO,não metropolitana,amazonia,246.08,Am,21558,2.24,93.68,56.86,Episodic/Epidemic
1,1100023,Ariquemes,11,RO,não metropolitana,amazonia,165.90,Am,100896,8.91,98.57,85.14,Epidemic
2,1100031,Cabixi,11,RO,não metropolitana,amazonia,224.18,Am,5107,21.01,95.49,49.77,Episodic/Epidemic
3,1100049,Cacoal,11,RO,metropolitana,amazonia,229.95,Am,92202,54.03,98.16,78.94,Epidemic
4,1100056,Cerejeiras,11,RO,não metropolitana,amazonia,192.22,Am,15237,16.73,98.62,78.35,Episodic/Epidemic


Criando o cabeçario da tabela com o número de municípios em cada uma das classes: 

In [5]:
df_cab = pd.DataFrame(df_det.dengue_pattern.value_counts()).T

df_cab = df_cab[prof_order]

df_cab

dengue_pattern,No Transmission,Episodic,Episodic/Epidemic,Epidemic,Persistent Transmission
count,208,1780,2994,574,14


Aplicando o test chi2 na tabela com o bioma: 

In [6]:
c_table_bioma_count = pd.crosstab(df_det['bioma'], df_det['dengue_pattern'])[prof_order]

chi2, p, _, _f = stats.chi2_contingency(c_table_bioma_count)

# calculando a frequência 
c_table_bioma_freq = (100*(c_table_bioma_count/df_cab.values[0])).round(2)

# somando como string a contagem e frequências para atingir o formato desejado 
c_table_bioma_end = c_table_bioma_count.astype(str) + ' (' + c_table_bioma_freq.astype(str) + ')'

# adicionando informações extras na tabela: 
c_table_bioma_end['type'] = 'bioma'

c_table_bioma_end['p-value'] = p

c_table_bioma_end

dengue_pattern,No Transmission,Episodic,Episodic/Epidemic,Epidemic,Persistent Transmission,type,p-value
bioma,,,,,,,
amazonia,9 (4.33),174 (9.78),269 (8.98),51 (8.89),0 (0.0),bioma,1.091618e-98
caatinga,8 (3.85),291 (16.35),721 (24.08),75 (13.07),1 (7.14),bioma,1.091618e-98
cerrado,5 (2.4),265 (14.89),654 (21.84),135 (23.52),4 (28.57),bioma,1.091618e-98
mata atlantica,143 (68.75),955 (53.65),1322 (44.15),310 (54.01),9 (64.29),bioma,1.091618e-98
pampa,43 (20.67),94 (5.28),22 (0.73),1 (0.17),0 (0.0),bioma,1.091618e-98
pantanal,0 (0.0),1 (0.06),6 (0.2),2 (0.35),0 (0.0),bioma,1.091618e-98


Aplicando o teste chi2 na tabela dos tipos climáticos: 

In [7]:
c_table_tipo_count = pd.crosstab(df_det['koppen'], df_det['dengue_pattern'])

chi2, p, _, _f = stats.chi2_contingency(c_table_tipo_count)

c_table_tipo_freq = (100*(c_table_tipo_count/df_cab.values[0])).round(2)

c_table_tipo_end = c_table_tipo_count.astype(str) + ' (' + c_table_tipo_freq.astype(str) + ')'

c_table_tipo_end['type'] = 'tipos climáticos'

c_table_tipo_end['p-value'] = p


c_table_tipo_end

dengue_pattern,Epidemic,Episodic,Episodic/Epidemic,No Transmission,Persistent Transmission,type,p-value
koppen,,,,,,,
Af,38 (18.27),75 (4.21),105 (3.51),2 (0.35),0 (0.0),tipos climáticos,9.003411e-147
Am,58 (27.88),101 (5.67),230 (7.68),4 (0.7),3 (21.43),tipos climáticos,9.003411e-147
As,68 (32.69),264 (14.83),552 (18.44),7 (1.22),1 (7.14),tipos climáticos,9.003411e-147
Aw,175 (84.13),295 (16.57),856 (28.59),8 (1.39),3 (21.43),tipos climáticos,9.003411e-147
BSh,29 (13.94),107 (6.01),283 (9.45),1 (0.17),0 (0.0),tipos climáticos,9.003411e-147
Cfa,103 (49.52),460 (25.84),459 (15.33),114 (19.86),2 (14.29),tipos climáticos,9.003411e-147
Cfb,24 (11.54),214 (12.02),100 (3.34),66 (11.5),1 (7.14),tipos climáticos,9.003411e-147
Cwa,61 (29.33),99 (5.56),243 (8.12),3 (0.52),1 (7.14),tipos climáticos,9.003411e-147
Cwb,18 (8.65),165 (9.27),166 (5.54),3 (0.52),3 (21.43),tipos climáticos,9.003411e-147


In [8]:
df_cab['type'] = ''
df_cab['p-value'] = ''

Gerando a tabela final concatenando os cabeçarios e os tipos climático: 

In [9]:
df_end_cat = pd.concat([df_cab, c_table_bioma_end, c_table_tipo_end])

df_end_cat.to_csv('tabela_categorica.csv')

df_end_cat

dengue_pattern,No Transmission,Episodic,Episodic/Epidemic,Epidemic,Persistent Transmission,type,p-value
count,208,1780,2994,574,14,,
amazonia,9 (4.33),174 (9.78),269 (8.98),51 (8.89),0 (0.0),bioma,0.0
caatinga,8 (3.85),291 (16.35),721 (24.08),75 (13.07),1 (7.14),bioma,0.0
cerrado,5 (2.4),265 (14.89),654 (21.84),135 (23.52),4 (28.57),bioma,0.0
mata atlantica,143 (68.75),955 (53.65),1322 (44.15),310 (54.01),9 (64.29),bioma,0.0
pampa,43 (20.67),94 (5.28),22 (0.73),1 (0.17),0 (0.0),bioma,0.0
pantanal,0 (0.0),1 (0.06),6 (0.2),2 (0.35),0 (0.0),bioma,0.0
Af,2 (0.35),75 (4.21),105 (3.51),38 (18.27),0 (0.0),tipos climáticos,0.0
Am,4 (0.7),101 (5.67),230 (7.68),58 (27.88),3 (21.43),tipos climáticos,0.0
As,7 (1.22),264 (14.83),552 (18.44),68 (32.69),1 (7.14),tipos climáticos,0.0


Abrindo o dataset com os dados climáticos: 

In [10]:
df = pd.read_csv('../Determinantes/clima/clima_agg_10_19.csv', index_col = 'Unnamed: 0')

df.loc[:, 'dengue_pattern'] = df.dengue_pattern.replace(update_prof)

df.head()

,geocodigo,temp_med,umid_med,precip_tot,year,muni_code,dengue_pattern
0,2700102,25.917411,71.160050,174.737289,2010,2700102,Episodic/Epidemic
1,2700201,25.484200,80.407899,424.205669,2010,2700201,Episodic/Epidemic
2,2700300,25.588142,77.216115,277.945870,2010,2700300,Epidemic
3,2700409,24.897218,81.989589,423.138619,2010,2700409,Episodic/Epidemic
4,2700508,25.800147,78.780638,491.421552,2010,2700508,Episodic


In [11]:
prof_order = ['No Transmission', 'Episodic', 'Episodic/Epidemic', 'Epidemic', 
       'Persistent Transmission']

In [12]:

def mean_std(df, col = 'temp_med'):
    '''
    Essa função retorna a media e desvio padrão no formato de string para cada um dos grupos, 
    além do nome da coluna utilizada e do p-valor obtido no teste anova 
    '''
    
    no_trans = df.loc[df.dengue_pattern == 'No Transmission'][col].values
    
    epis = df.loc[df.dengue_pattern == 'Episodic'][col].values
    
    epis_epid = df.loc[df.dengue_pattern == 'Episodic/Epidemic'][col].values
    
    epid = df.loc[df.dengue_pattern == 'Epidemic'][col].values
    
    per_trans = df.loc[df.dengue_pattern ==  'Persistent Transmission'][col].values
    
    f_stat, p = stats.f_oneway(no_trans, epis, epis_epid, epid, per_trans)

    return [f'{round(np.mean(no_trans),2)} ({round(np.std(no_trans),2)})',
           f'{round(np.mean(epis),2)} ({round(np.std(epis),2)})',
           f'{round(np.mean(epis_epid),2)} ({round(np.std(epis_epid),2)})',
           f'{round(np.mean(epid),2)} ({round(np.std(epid),2)})',
           f'{round(np.mean(per_trans),2)} ({round(np.std(per_trans),2)})',
            col, p]

# o método abaixo contatena os valores obtidos pela função mean_std no formato de um dataframe com o valor obtido para todas as variáveis 
# numéricas 

num_values = pd.concat([pd.DataFrame([mean_std(df, col = 'temp_med')], columns = ['No Transmission', 'Episodic', 'Episodic/Epidemic', 'Epidemic', 
       'Persistent Transmission', 'column', 'p-value']),
                            pd.DataFrame([mean_std(df, col = 'umid_med')], columns = ['No Transmission', 'Episodic', 'Episodic/Epidemic', 'Epidemic', 
       'Persistent Transmission', 'column', 'p-value']),
                            pd.DataFrame([mean_std(df, col = 'precip_tot')], columns = ['No Transmission', 'Episodic', 'Episodic/Epidemic', 'Epidemic', 
       'Persistent Transmission', 'column', 'p-value']),
                       pd.DataFrame([mean_std(df_det, col = 'altitude')], columns = ['No Transmission', 'Episodic', 'Episodic/Epidemic', 'Epidemic', 
       'Persistent Transmission', 'column', 'p-value']),
                       pd.DataFrame([mean_std(df_det, col = 'pop22')], columns = ['No Transmission', 'Episodic', 'Episodic/Epidemic', 'Epidemic', 
       'Persistent Transmission', 'column', 'p-value']),
                       pd.DataFrame([mean_std(df_det, col = 'prop_esgot')], columns = ['No Transmission', 'Episodic', 'Episodic/Epidemic', 'Epidemic', 
       'Persistent Transmission', 'column', 'p-value']),
                       pd.DataFrame([mean_std(df_det, col = 'prop_abast')], columns = ['No Transmission', 'Episodic', 'Episodic/Epidemic', 'Epidemic', 
       'Persistent Transmission', 'column', 'p-value']),
                       pd.DataFrame([mean_std(df_det, col = 'prop_residuo')], columns = ['No Transmission', 'Episodic', 'Episodic/Epidemic', 'Epidemic', 
       'Persistent Transmission', 'column', 'p-value'])],
                           ignore_index = True).set_index('column')


num_values

,No Transmission,Episodic,Episodic/Epidemic,Epidemic,Persistent Transmission,p-value
column,,,,,,
temp_med,19.7 (2.96),23.01 (3.66),24.39 (2.48),24.13 (2.15),22.9 (2.04),0.000000e+00
umid_med,78.08 (5.75),74.58 (7.97),71.76 (7.3),73.02 (7.02),72.68 (5.34),0.000000e+00
precip_tot,537.61 (199.32),431.99 (225.74),362.45 (190.58),399.18 (187.07),386.26 (121.27),0.000000e+00
altitude,483.08 (310.99),454.07 (322.61),449.65 (262.99),435.87 (269.47),586.15 (328.86),1.070640e-01
pop22,4778.05 (4016.39),10371.43 (11496.76),22390.51 (34541.02),150761.91 (238590.8),2480055.43 (3116401.34),0.000000e+00
prop_esgot,41.52 (24.77),35.71 (28.05),42.16 (32.03),62.16 (30.04),87.2 (11.4),1.794015e-74
prop_abast,86.09 (12.97),82.39 (16.66),86.2 (14.39),94.0 (8.15),98.97 (0.89),2.441696e-60
prop_residuo,65.33 (23.12),62.3 (23.54),71.53 (19.78),88.74 (11.86),99.17 (0.67),4.233893e-157


In [13]:
df_end_num = pd.concat([df_cab.drop('type', axis =1), num_values])

df_end_num.to_csv('tabela_numerica.csv')

df_end_num

,No Transmission,Episodic,Episodic/Epidemic,Epidemic,Persistent Transmission,p-value
count,208,1780,2994,574,14,
temp_med,19.7 (2.96),23.01 (3.66),24.39 (2.48),24.13 (2.15),22.9 (2.04),0.0
umid_med,78.08 (5.75),74.58 (7.97),71.76 (7.3),73.02 (7.02),72.68 (5.34),0.0
precip_tot,537.61 (199.32),431.99 (225.74),362.45 (190.58),399.18 (187.07),386.26 (121.27),0.0
altitude,483.08 (310.99),454.07 (322.61),449.65 (262.99),435.87 (269.47),586.15 (328.86),0.107064
pop22,4778.05 (4016.39),10371.43 (11496.76),22390.51 (34541.02),150761.91 (238590.8),2480055.43 (3116401.34),0.0
prop_esgot,41.52 (24.77),35.71 (28.05),42.16 (32.03),62.16 (30.04),87.2 (11.4),0.0
prop_abast,86.09 (12.97),82.39 (16.66),86.2 (14.39),94.0 (8.15),98.97 (0.89),0.0
prop_residuo,65.33 (23.12),62.3 (23.54),71.53 (19.78),88.74 (11.86),99.17 (0.67),0.0


In [14]:
df_det.loc[df_det.dengue_pattern == 'No Transmission']['altitude'].mean()

483.0828365384616

In [15]:
df_det.loc[df_det.dengue_pattern == 'Epidemic']['prop_abast'].mean()

94.00118466898955

In [16]:
df_det.loc[df_det.dengue_pattern == 'Epidemic']['prop_abast'].std()

8.155499583913915